# Recursion in Python 

Recursion in Python is a programming technique where a function calls itself directly or indirectly in order to solve a problem.

It works by breaking a complex problem down into smaller, simpler sub-problems that are identical in nature to the original problem. This is often summarized by the phrase: *"To solve a big problem, solve a smaller version of the exact same problem."*

---

## The Two Essential Parts of Recursion
Every recursive function must have these two components:

1.  **The Base Case (Stopping Condition):** The condition under which the function **stops** calling itself. This prevents infinite loops and provides a concrete answer for the smallest version of the problem. *If you forget this, your program will run until it hits the recursion limit.*

2.  **The Recursive Case:** The part where the function calls itself with a modified argument, steadily moving closer to the base case.

---

## Types of Recursion
Beyond the basic direct recursion, there are other patterns we should be aware of:

-   **Direct Recursion:** The function calls itself directly (the standard type covered above).

-   **Indirect Recursion (Mutual Recursion):** Function `A` calls function `B`, and function `B` calls function `A`. This is valid in Python, though less common, and both functions must have their own base cases to terminate.

-   **Linear vs. Tree Recursion:**

    -   *Linear:* The function calls itself at most once per invocation (e.g., calculating a factorial).

    -   *Tree/Branching:* The function calls itself multiple times per invocation (e.g., traversing a binary tree or generating Fibonacci numbers), leading to exponential growth in the number of calls.

---



## How It Works (The Call Stack)
When a recursive function calls itself, Python pushes the current function call onto a **call stack** (LIFO - Last In, First Out).

-   Each call waits for the recursive call beneath it to finish.
-   Each call maintains its own local variables and state, which are isolated from the other calls.
-   Once the base case is hit, Python starts "unwinding" the stack, returning values back up to the original caller.

**Conceptual execution flow:**  
`function(5)` → calls `function(4)` → calls `function(3)` → ... → hits the base case → returns values back up the chain.

---

## Important Python-Specific Limitations & Risks

-   **Recursion Limit:** To prevent a native OS stack overflow (which would crash the Python interpreter entirely), Python enforces a default recursion limit of **1000** calls. If you try to recurse beyond that, you will get a `RecursionError: maximum recursion depth exceeded`. You can technically change this limit using `sys.setrecursionlimit()`, but doing so is highly risky and can cause segmentation faults if you push it too far.

-   **No Tail-Call Optimization (TCO):** In some languages (like Haskell or Scheme), if the recursive call is the very last operation in the function (tail position), the compiler optimizes it to use a loop (saving memory).
**Python explicitly does not support TCO**, as Guido van Rossum (Python's creator) considered it unpythonic. Therefore, every single recursive call consumes memory on the stack, making recursion inherently memory-heavy.

-   **Risk of Exponential Complexity:** If you write a branching recursive function without caching (memoization), it can recalculate the same sub-problems thousands of times, leading to horrifically slow execution times.

---

## The Underlying Logic: Mathematical Induction
Recursion in programming is fundamentally based on **Mathematical Induction**.

-   The **Base Case** is the equivalent of the "base step" in induction (proving it works for the smallest value).
-   The **Recursive Case** is the "inductive step" (assuming it works for `n-1` to prove it works for `n`).
Understanding this connection helps you write correct recursive functions by focusing on *how* to break the problem down, rather than trying to mentally trace every single step.

---

## Common Conceptual Use Cases (Without Code)
Recursion is particularly suited for problems with a naturally hierarchical or nested structure:

-   **Tree and Graph Traversals:** Depth-First Search (DFS), parsing HTML/XML/JSON data.
-   **Backtracking Algorithms:** Solving mazes, the N-Queens puzzle, or Sudoku.
-   **Divide and Conquer Algorithms:** QuickSort, MergeSort (though iterative versions exist, recursive ones are easier to conceptualize).
-   **File System Navigation:** Recursively listing all files and subfolders within a directory.
-   **Mathematical Sequences:** Calculating Fibonacci numbers or factorial values.

---



## The "Iterative Alternative" Rule
**Crucial fact:** Every recursive function can be rewritten iteratively using a loop and an explicit manual stack (a `list` in Python acting as a stack). 

-   Doing this often uses less memory (avoids the call stack limit) and runs faster.
-   However, it usually results in much more complex and harder-to-read code, especially for tree structures.

---

## When to Use Recursion vs. Loops

| Aspect | Recursion | Iteration (Loops) |
| :--- | :--- | :--- |
| **Memory Usage** | High (O(n) stack space) | Low (O(1) space) |
| **Speed** | Slower (due to function call overhead) | Faster |
| **Code Readability** | Extremely elegant for branching structures (trees, graphs, divide-and-conquer). Often 3–4 lines of code. | Elegant for simple, linear repetitions. Can become overly complex for nested data. |
| **Risk** | Stack overflow, RecursionError, exponential recomputation. | Infinite loop (if condition is wrong). |
| **Best Use Case** | Navigating deep, nested, or unknown hierarchical data where depth is typically small (e.g., < 50 levels). | Simple sequential tasks, heavy mathematical computations, or when performance and memory are critical. |

---

## The Golden Rule for Python
**"Flat is better than nested." (The Zen of Python)** – If you can do it cleanly with a loop, do it with a loop. Only use recursion when the problem is *naturally* hierarchical (like navigating a file system or parsing nested structures). 

**Professional Tip:** If your recursion depth is likely to exceed 20–30 levels in a production environment, you should strongly consider rewriting your solution iteratively or using an explicit `stack` data structure to simulate recursion manually. Defensive programming dictates that you should never rely on recursive calls that approach the 1000 limit.

In [19]:
n=200000
result = 1
for i in range(2, n+1):
    result *= i
 

In [24]:
def factorial(n):
    if n<=1:
        return 1
    return n * factorial(n-1)


result_fac = factorial(10000)

RecursionError: maximum recursion depth exceeded

In [2]:
import time
import tracemalloc
import sys

# --- 1. Define the Functions ---

def factorial_iterative(n):
    """Iterative approach using a loop (O(1) space)."""
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

def factorial_recursive(n):
    """Recursive approach (O(n) stack space)."""
    if n <= 1:  # Base case
        return 1
    return n * factorial_recursive(n - 1)  # Recursive case


# --- 2. The Measurement Wrapper ---

def measure_performance(func, n, label):
    """
    Runs a single function, measures time (perf_counter) 
    and peak memory usage (tracemalloc).
    """
    # Start tracking memory
    tracemalloc.start()
    # Record initial peak memory (to calculate delta later)
    _, peak_before = tracemalloc.get_traced_memory()

    # Measure execution time
    start_time = time.perf_counter()
    result = func(n)
    end_time = time.perf_counter()

    # Get memory usage after execution
    _, peak_after = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Calculate metrics
    elapsed_ms = (end_time - start_time) * 1000
    memory_used_kb = (peak_after - peak_before) / 1024

    # Print results
    print(f"--- {label} ---")
    print(f"n = {n}")
    print(f"Result (last 6 digits): {result % 1000000}")  # Avoid printing giant numbers
    print(f"Time taken: {elapsed_ms:.4f} milliseconds")
    print(f"Peak memory increase: {memory_used_kb:.2f} KB")
    print("-" * 40)


# --- 3. Run the Experiments ---

if __name__ == "__main__":
    # Set a higher recursion limit JUST IN CASE (safe for n=800)
    sys.setrecursionlimit(5000)

    # --- Test 1: Small number (n=20) ---
    print("\n🔬 EXPERIMENT 1: SMALL INPUT (n=20)\n")
    measure_performance(factorial_iterative, 20, "Iterative (Small)")
    measure_performance(factorial_recursive, 20, "Recursive (Small)")

    # --- Test 2: Large number (n=800) ---
    # 800 is safe (under the 1000 default limit) but large enough to stress the stack.
    print("\n🔬 EXPERIMENT 2: LARGE INPUT (n=800)\n")
    measure_performance(factorial_iterative, 800, "Iterative (Large)")
    measure_performance(factorial_recursive, 800, "Recursive (Large)")


🔬 EXPERIMENT 1: SMALL INPUT (n=20)

--- Iterative (Small) ---
n = 20
Result (last 6 digits): 640000
Time taken: 0.0133 milliseconds
Peak memory increase: 0.12 KB
----------------------------------------
--- Recursive (Small) ---
n = 20
Result (last 6 digits): 640000
Time taken: 0.0215 milliseconds
Peak memory increase: 0.07 KB
----------------------------------------

🔬 EXPERIMENT 2: LARGE INPUT (n=800)

--- Iterative (Large) ---
n = 800
Result (last 6 digits): 0
Time taken: 1.0177 milliseconds
Peak memory increase: 1.84 KB
----------------------------------------
--- Recursive (Large) ---
n = 800
Result (last 6 digits): 0
Time taken: 4.2849 milliseconds
Peak memory increase: 17.46 KB
----------------------------------------


In [47]:
# palindrome using recursion

def palindrome(string):
    
    if len(string)<=1:
        return True
    else:
        if string[0] == string[-1]:
            return palindrome(string[1:-1])
        else:
            return False

s1 = 'madam'
s2 = 'Arbaz'
s3 = 'abba'

print(f"IS {s1} a Palindrome : {palindrome(s1)}")
print(f"IS {s2} a Palindrome : {palindrome(s2)}")
print(f"IS {s3} a Palindrome : {palindrome(s3)}")

IS madam a Palindrome : True
IS Arbaz a Palindrome : False
IS abba a Palindrome : True


In [64]:
## Rabbit Problem (Fibonacci Sequence)
import time
def fib(n):
    
    if n<=2:
        return 1
    else:
        return fib(n-1) + fib(n-2)

month=36

start_time = time.time()
print(f"Number of rabbits at the end of {month} Month is : {fib(month)}")
print(f"Time Taken to complete the code : {time.time() - start_time}")

# This recursion is non-linear and highly efficient with time complexity O(2^n)

Number of rabbits at the end of 36 Month is : 14930352
Time Taken to complete the code : 0.5789718627929688


In [70]:
# So let's solve the same problem by implementing a technique called 'Memoization'. Instead of calculating the same thing will store that result and use on the go.

def memo_fib(n, memory):
    
    if n in memory:
        return memory[n]
    else:
        memory[n] = memo_fib(n-1, memory) + memo_fib(n-2, memory)
        return memory[n]

memory = {1:1, 2:1}

month=50

start_time = time.time()
print(f"Number of rabbits at the end of {month} Month is : {memo_fib(month, memory)}")
print(f"Time Taken to complete the code : {time.time() - start_time}")



Number of rabbits at the end of 50 Month is : 12586269025
Time Taken to complete the code : 0.00012803077697753906
